# 05 — Association Rules: Pola Pembelian (FP-Growth)

Notebook ini menemukan produk yang sering dibeli bersamaan menggunakan algoritma FP-Growth dari Spark MLlib.

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.insert(0, "/home/jovyan/work")

from analysis.spark_session import create_spark_session
spark = create_spark_session("05 - Association Rules")

## 2. Baca Data

In [ ]:
BUCKET = "datalake"

df_orders   = spark.read.csv(f"s3a://{BUCKET}/raw/rdbms/orders/*/orders_from_db.csv",     header=True, inferSchema=True)
df_products = spark.read.csv(f"s3a://{BUCKET}/raw/xlsx/products/*/products_from_xlsx.csv", header=True, inferSchema=True)

print(f"Orders   : {df_orders.count()} baris")
print(f"Products : {df_products.count()} baris")

## 3. Bangun Format Transaksi

In [ ]:
from analysis.preprocessing import build_transaction_items

df_transactions = build_transaction_items(df_orders, df_products)

print("Format transaksi per pelanggan:")
df_transactions.show(5, truncate=False)

## 4. Jalankan FP-Growth

In [ ]:
from analysis.mining.association import run_fpgrowth

# min_support=0.1 → itemset harus muncul di ≥ 10% transaksi (~3 pelanggan dari 30)
# min_confidence=0.5 → P(B|A) minimal 50%
df_freq_items, df_rules = run_fpgrowth(
    df_transactions,
    items_col="items",
    min_support=0.1,
    min_confidence=0.5,
)

print(f"Frequent itemsets ditemukan : {df_freq_items.count()}")
print(f"Association rules ditemukan : {df_rules.count()}")

## 5. Frequent Itemsets

In [ ]:
print("=== Top Frequent Itemsets ===")
df_freq_items.orderBy("freq", ascending=False).show(15, truncate=False)

## 6. Association Rules

In [ ]:
print("=== Association Rules (diurutkan berdasarkan lift) ===")
df_rules.select("antecedent", "consequent", "confidence", "lift") \n    .orderBy("lift", ascending=False) \n    .show(20, truncate=False)

## 7. Visualisasi Top Rules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pdf_rules = df_rules.toPandas()
pdf_rules["rule"] = pdf_rules["antecedent"].apply(lambda x: str(x)) + " → " + pdf_rules["consequent"].apply(lambda x: str(x))
pdf_rules_top = pdf_rules.sort_values("lift", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(pdf_rules_top["rule"], pdf_rules_top["lift"], color="steelblue")
ax.set_xlabel("Lift")
ax.set_title("Top 10 Association Rules berdasarkan Lift")
ax.axvline(x=1, color="red", linestyle="--", label="Lift = 1 (tidak ada asosiasi)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Simpan Rules ke MinIO

In [ ]:
from pyspark.sql import functions as F

# Konversi kolom Array ke String agar kompatibel dengan format CSV
df_rules_csv = df_rules.withColumn(
    "antecedent", F.concat_ws(", ", F.col("antecedent"))
).withColumn(
    "consequent", F.concat_ws(", ", F.col("consequent"))
)

df_rules_csv.write.mode("overwrite").option("header", True) \
    .csv(f"s3a://{BUCKET}/processed/association_rules/")
print("Association rules disimpan ke processed zone.")